# Phase 0-1 — BlenderProc smoke + light_vehicle render (Colab T4)

**Важливо:** BlenderProc запускається ТІЛЬКИ через `!blenderproc run <script>`. `import blenderproc` у Python kernel ламається (це by design).

Працює напряму з Google Drive. Без git/GitHub.

**Перед запуском:**
1. Runtime → Change runtime type → **T4 GPU**.
2. У Drive має бути `MyDrive/yolo_bb/datasetforge/` з assets+engine+configs (drag&drop усю папку datasetforge/ з твого комп'ютера).

**Gates:**
- G0: `blenderproc quickstart` → не пустий PNG.
- G1: 20 light_vehicle frames + bbox overlay sanity.

In [ ]:
# 1. Mount Drive — браузер попросить дозвіл при першому запуску.
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/yolo_bb'
REPO  = f'{DRIVE}/datasetforge'
ASSETS = f'{REPO}/assets'
assert os.path.isdir(REPO), f'Не знайдено {REPO}. Залий папку datasetforge/ у Drive/MyDrive/yolo_bb/'
print('repo at:', REPO)
!ls $REPO

In [ ]:
# 2. Install BlenderProc + bpycv. 
# УВАГА: НЕ робимо `import blenderproc` тут — це by design не дозволено.
# Перевірка установки через CLI.
!pip install -q blenderproc bpycv
!blenderproc --help | head -5

In [ ]:
# 3. Перевір що моделі і assets на місці
print('-- models/light_vehicle --')
!ls $ASSETS/models/light_vehicle/ 2>/dev/null
print('-- models/ifv_apc --')
!ls $ASSETS/models/ifv_apc/ 2>/dev/null
print('-- hdri --')
!ls $ASSETS/hdri/*/ 2>/dev/null | head -20
print('-- textures --')
!ls $ASSETS/textures/ground/*/ 2>/dev/null | head -20

## G0 — BlenderProc quickstart sanity

Перший запуск довантажить Blender 4.x (~500 MB, ~3 хв). PNG згенерується у `/content/quickstart_output/`.

In [ ]:
# 4. G0: quickstart — рендерить дефолтну BlenderProc сцену
# MPLBACKEND=agg fix: Colab ставить inline backend, Blender'овий matplotlib його не розуміє
import os
os.environ['MPLBACKEND'] = 'agg'
!MPLBACKEND=agg blenderproc quickstart

In [ ]:
# 4b. Витягнути картинку з BlenderProc HDF5 output (quickstart зберігає colors+depth+seg у hdf5)
import glob
import h5py
import numpy as np
from PIL import Image

hdf5_files = sorted(glob.glob('/content/output/*.hdf5') + glob.glob('output/*.hdf5'))
print('found HDF5:', hdf5_files[:3])

if hdf5_files:
    with h5py.File(hdf5_files[0], 'r') as f:
        print('keys:', list(f.keys()))
        colors = np.array(f['colors'])
    print('shape:', colors.shape, 'dtype:', colors.dtype)
    img = Image.fromarray(colors.astype(np.uint8))
    img.thumbnail((512, 512))
    display(img)
else:
    print('UWAGA: HDF5 не знайдено. Перевір що quickstart завершився успішно.')

## G1 — Render 20 light_vehicle frames

Запускаємо наш `render_runner.py` через `blenderproc run`. Він підтягне config + моделі з Drive.

In [ ]:
# 5. Render light_vehicle smoke (20 кадрів). 
# blenderproc run САМ запустить Blender + наш скрипт.
# cwd має бути $DRIVE щоб `datasetforge.engine.*` імпортувались.
import os
os.chdir(DRIVE)
os.environ['MPLBACKEND'] = 'agg'  # fix matplotlib backend для Blender Python
OUT = '/content/lv_smoke'
!MPLBACKEND=agg blenderproc run $REPO/engine/render_runner.py \
    --config $REPO/configs/v1_light_vehicle.yaml \
    --n 20 \
    --out $OUT \
    --assets-root $ASSETS

In [ ]:
# 6. Перевірка output
import glob
imgs = sorted(glob.glob(f'{OUT}/images/*.jpg'))
lbls = sorted(glob.glob(f'{OUT}/labels/*.txt'))
metas = sorted(glob.glob(f'{OUT}/metadata/*.json'))
print(f'images: {len(imgs)}  labels: {len(lbls)}  metadata: {len(metas)}')
from PIL import Image
if imgs:
    display(Image.open(imgs[0]))
else:
    print('UWAGA: жодного кадру. Перевір output клітинки 5.')

In [ ]:
# 7. Sanity overlay bbox на 5 random кадрах
import random
from PIL import Image, ImageDraw
from pathlib import Path

sample = random.sample(imgs, min(5, len(imgs))) if imgs else []
for img_path in sample:
    p = Path(img_path)
    lbl = Path(OUT) / 'labels' / (p.stem + '.txt')
    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    w, h = img.size
    if lbl.exists():
        for line in lbl.read_text().strip().splitlines():
            cls, xc, yc, bw, bh = map(float, line.split())
            x0 = (xc - bw / 2) * w
            y0 = (yc - bh / 2) * h
            x1 = (xc + bw / 2) * w
            y1 = (yc + bh / 2) * h
            draw.rectangle([x0, y0, x1, y1], outline='lime', width=2)
    display(img)

In [ ]:
# 8. Backup output до Drive (щоб не загубилось після disconnect)
!rsync -a $OUT/ $DRIVE/lv_smoke_v1/
!ls $DRIVE/lv_smoke_v1